<a href="https://colab.research.google.com/github/DrKroll/llm-humanidades-salamanca/blob/main/notebooks/taller_llm/02_tokenizador.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 02 · El tokenizador: de texto a números

**Objetivo:** entrenar un tokenizador **BPE** (*Byte Pair Encoding*) con tus textos y ver cómo trocea el lenguaje.

Un ordenador no entiende letras, solo números. El **tokenizador** divide el texto en trozos
(**tokens**) y asigna un número a cada uno. Opciones:

| Unidad | Ejemplo con «caballero» | Problema |
|---|---|---|
| letras | c·a·b·a·l·l·e·r·o | secuencias muy largas |
| palabras | caballero | vocabulario enorme, palabras nuevas desconocidas |
| **subpalabras (BPE)** | caball·ero | ✅ el compromiso que usan GPT, Llama, etc. |

**Cómo funciona BPE:** empieza con caracteres sueltos y une una y otra vez el par más frecuente
(«q»+«u» → «qu», «qu»+«e» → «que»...) hasta alcanzar el tamaño de vocabulario deseado.

In [ ]:
# ⚙️ Preparación: ejecuta esta celda primero (Mayús + Intro)
import os, sys
REPO_URL = "https://github.com/DrKroll/llm-humanidades-salamanca"
REPO_DIR = "llm-humanidades-salamanca"

if "google.colab" in sys.modules:
    # En Colab: descargamos el repositorio e instalamos el paquete del taller
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL}
    os.chdir(REPO_DIR)
    !pip install -q -e .
else:
    # En tu ordenador: subimos de carpeta hasta encontrar config.yaml
    while not os.path.exists("config.yaml") and os.getcwd() != "/":
        os.chdir("..")
    if os.path.abspath("src") not in sys.path:
        sys.path.insert(0, os.path.abspath("src"))
print("Carpeta de trabajo:", os.getcwd())

## ⚙️ Ajustes

In [ ]:
from dhllm import load_config
cfg = load_config()

CARPETAS = cfg.data.folders
VOCABULARIO = cfg.tokenizer.vocab_size   # prueba 500, 2000, 8000
FRASE = "En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho tiempo que vivía un hidalgo."

## 1. Cargar textos y entrenar el tokenizador
Tarda unos segundos.

In [ ]:
from dhllm.data import load_texts
from dhllm.tokenizer import train_bpe
textos = load_texts(CARPETAS)
tok = train_bpe(textos, vocab_size=VOCABULARIO, min_frequency=cfg.tokenizer.min_frequency)

## 2. Ver los tokens
Cada color es un token. El espacio suele ir pegado al principio de la palabra.

In [ ]:
from dhllm.tokenizer import show_tokens
show_tokens(tok, FRASE);

Y estos son los números que «ve» el modelo:

In [ ]:
ids = tok.encode(FRASE)
print(ids)
print("De vuelta a texto:", tok.decode(ids))

## 3. ¿Qué ha aprendido el tokenizador?
Los tokens más largos del vocabulario suelen ser palabras frecuentes de tu corpus.

In [ ]:
from dhllm.tokenizer import show_vocabulary_sample
show_vocabulary_sample(tok, n=40);

## 4. Comparar con un tokenizador «de verdad»
Comparamos con el tokenizador de GPT-2 (entrenado sobre todo con inglés) y con el de SmolLM2.
**Menos tokens por palabra = más eficiente** para ese idioma. (Necesita conexión a internet.)

In [ ]:
from dhllm.tokenizer import compare_tokenizers
otros = {"nuestro BPE": tok}
try:
    from transformers import AutoTokenizer
    otros["GPT-2 (inglés)"] = AutoTokenizer.from_pretrained("gpt2")
    otros["SmolLM2"] = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
except Exception as e:
    print("No se pudieron descargar otros tokenizadores:", e)
compare_tokenizers(FRASE, otros);

In [ ]:
if "GPT-2 (inglés)" in otros:
    show_tokens(otros["GPT-2 (inglés)"], FRASE);

## 5. Guardar el tokenizador
Lo usaremos en el cuaderno 03.

In [ ]:
tok.save(cfg.tokenizer.save_dir)

## 🖐️ Tu turno
1. Prueba `VOCABULARIO = 300` y `VOCABULARIO = 10000`. ¿Qué cambia en los colores de la frase?
2. Tokeniza una palabra larga: `show_tokens(tok, "anticonstitucionalmente")`.
3. Tokeniza una frase en otro idioma (latín, inglés, catalán...) o con ortografía antigua. ¿Qué pasa?
4. **Para pensar:** si un idioma necesita el doble de tokens que el inglés, ¿qué consecuencias tiene
   en el coste y la calidad de los grandes modelos comerciales?